# Overview
The goal of this study is to evaluate and compare different clustering algorithms.
it consists of 4 key parts:
1. Loading datasets
2. Clustering with each clustering algorithm
3. Measuring performance
4. Summary


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import clustbench
import genieclust 
import pandas as pd


# Loading datasets

In [4]:
from utils.loader import load_datasets
from utils.preprocess import preprocess_datasets
datasets = load_datasets()
# removing noise
preprocessed_datasets = preprocess_datasets(datasets)


Loading datasets from graves: 100%|██████████| 10/10 [00:00<00:00, 617.28it/s]


# Performing clustering

In [5]:
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from genieclust import Genie
from functools import partial

model_configs = [
    ("KMeans", KMeans),
    ("Genie_g0.1", partial(Genie, 
                    gini_threshold=0.1)),
    ("Genie_g0.3", partial(Genie, 
                    gini_threshold=0.3)),
    ("Genie_g0.5", partial(Genie, 
                    gini_threshold=0.5)),
    ("Genie_g0.7", partial(Genie, gini_threshold=0.7)),
    ("Genie_g0.9", partial(Genie, gini_threshold=0.9)),
    ("AHC_ward", partial(AgglomerativeClustering, 
                    linkage='ward')),
    ("AHC_single", partial(AgglomerativeClustering,
                     linkage='single')),
    ("AHC_average", partial(AgglomerativeClustering, 
                    linkage='average')),
    ("AHC_complete", partial(AgglomerativeClustering, 
                    linkage='complete')),
    ("SpectralClustering", partial(SpectralClustering, 
                            eigen_solver='arpack', 
                            affinity='nearest_neighbors', 
                            n_init=1,
                            random_state=42)),
    ("GaussianMixture", GaussianMixture)
]

In [6]:
from utils.clustering import  cluster_datasets_
results = {}
for model_name, model in model_configs:
    results[f'{model_name}'] = cluster_datasets_(preprocessed_datasets, model, model_name)


Clustering datasets with SpectralClustering:   0%|          | 0/43 [00:00<?, ?it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   2%|▏         | 1/43 [00:00<00:04,  8.97it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   5%|▍         | 2/43 [00:00<00:10,  3.96it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering dataset

Widać, ze K-Means i Genie są najszybsze


# Evaluating Created Clusters

In [47]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from genieclust.cluster_validity import calinski_harabasz_index, negated_davies_bouldin_index, silhouette_index
from genieclust.compare_partitions import compare_partitions
import tqdm

def evaluate_clustering(X, y_true, y_pred, nca = True):
    metrics = {
        'ARI': adjusted_rand_score(y_true, y_pred),
        'NMI': normalized_mutual_info_score(y_true, y_pred),
        'NCA' : compare_partitions( y_true, y_pred)['nca'] if nca else None,
        'Silhouette': silhouette_index(X, y_pred),
        'Calinski-Harabasz': calinski_harabasz_index(X, y_pred),
        'Generalised Dunn': negated_davies_bouldin_index(X, y_pred)
    }
    return metrics

scores = {}
for model_name in results.keys():
    model_scores = []
    for dataset_name in tqdm.tqdm(results[model_name].keys(), desc=f"Evaluating {model_name}"):
        Y_pred = results[model_name][dataset_name]['Y_pred']
        Y_true = preprocessed_datasets[dataset_name]['labels']
        X = preprocessed_datasets[dataset_name]['X']
        metrics = evaluate_clustering(X, Y_true, Y_pred)
        model_scores.append({'dataset': dataset_name, **metrics})
    scores[model_name] = pd.DataFrame(model_scores)


        

Evaluating GaussianMixture: 100%|██████████| 43/43 [00:00<00:00, 48.02it/s]


In [54]:
import os
for file in os.listdir('results'):
    os.remove(f'results/{file}')
for model_name, score_df in scores.items():
    
    score_df.to_csv(f'results/{model_name}.csv')

## DBSCAN


Due to its nature DBSCAN will be considered separately 


In [55]:
from utils.eval_dbscan import run_dbscan_evaluation
dbscan_results = {}
dbscan_scores = {}
for dataset_name, dataset_obj in tqdm.tqdm(datasets.items()):
    evaled = run_dbscan_evaluation(dataset_name, dataset_obj)
    if evaled is not None:
        dbscan_results[dataset_name] = evaled
        dbscan_scores[dataset_name] = evaluate_clustering(evaled['X'], evaled['y_true'], evaled['y_dbscan'], nca=False)
    
        


 26%|██▌       | 11/43 [00:00<00:01, 27.63it/s]

Dataset ('sipu', 'flame'): skipped DBSCAN found only 1 cluster


 56%|█████▌    | 24/43 [00:00<00:00, 30.69it/s]

Dataset ('uci', 'glass'): skipped DBSCAN found only 1 cluster
Dataset ('uci', 'ionosphere'): skipped DBSCAN found only 1 cluster


100%|██████████| 43/43 [00:01<00:00, 38.76it/s]


In [80]:
dbscan_scores_df = pd.DataFrame(dbscan_scores).T
scores['DBSCAN'] = dbscan_scores_df

for 3 of our datasets it has skipped 

# SUMMARY

In [81]:
import jinja2
summary_list = []

for algo_name, df in scores.items():
 
    avg_metrics = df[['ARI', 'NMI', 'NCA', 'Silhouette', 'Calinski-Harabasz', 'Generalised Dunn']].mean()

    avg_metrics['Algorithm'] = algo_name
    summary_list.append(avg_metrics)


df_summary = pd.DataFrame(summary_list).set_index('Algorithm')
df_summary = df_summary.sort_values(by='ARI', ascending=False)

print("Zbiorcze podsumowanie algorytmów (średnie wyniki):")
display(df_summary.style.highlight_max(color='lightgreen', axis=0))



Zbiorcze podsumowanie algorytmów (średnie wyniki):


,ARI,NMI,NCA,Silhouette,Calinski-Harabasz,Generalised Dunn
Algorithm,,,,,,
Genie_g0.3,0.751250,0.795565,0.795846,0.384722,3369.245806,-21.324104
Genie_g0.5,0.740494,0.796455,0.766120,0.374176,2621.723677,-21.310293
Genie_g0.1,0.700229,0.753531,0.775161,0.374724,3554.457927,-21.328769
SpectralClustering,0.676790,0.745279,0.778015,0.360438,2783.739431,-21.293529
Genie_g0.7,0.619876,0.721337,0.644948,0.292759,6015.588030,-inf
GaussianMixture,0.608720,0.654376,0.707224,0.439800,8085.639718,-0.942354
KMeans,0.531709,0.594288,0.686361,0.496379,8296.780087,-0.797728
DBSCAN,0.522321,0.607098,nan,0.288059,2144.564303,-0.764869
AHC_ward,0.521303,0.586396,0.662681,0.488762,8279.334354,-0.793733


In [ ]:
fr